# Day 3 Session 2 -- Deployment Exercises

In the demos you saw how to build a validated AI service, cache responses semantically, monitor requests with structured logging, route queries by complexity, and track costs per model. Now you will build these production patterns yourself -- starting with a multi-tier cache, then a safe service with error recovery, and finally a smart router with cost tracking.

**Exercises:**
1. Multi-Tier Caching System (combines exact-match + semantic caching) -- *Easy (Core)*
2. Request Validator with Error Recovery (validated service + monitoring)
3. Smart Model Router with Cost Tracking (routing + cost management)

**Optional Exercises:**
1. Cache with TTL and Eviction (medium)
2. Full Monitoring Dashboard (medium)
3. Production AI Service -- combines ALL 5 demo patterns (challenging)

In [ ]:
!pip install -q openai chromadb python-dotenv

---
## Setup

In [ ]:
import os
import time
import hashlib
from datetime import datetime, timedelta
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional, Dict, List
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

import openai
import numpy as np
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

print("All imports successful!")

---
## Recap from Demos

| Demo | Pattern | Key takeaway |
|------|---------|-------------|
| 1 | AI Service | Request validation via dataclasses, structured responses, error handling |
| 2 | Semantic Caching | Embedding similarity at threshold 0.90-0.92 matches paraphrases |
| 3 | Monitoring | Per-engagement structured logging, latency/token tracking |
| 4 | Model Routing | LLM-based complexity classification -> model selection |
| 5 | Cost Tracking | Per-model cost calculation, budget alerts at 80% |

You will now combine these patterns into working systems.

---
## Exercise 1 (Easy): Build a Multi-Tier Caching System

**Reference:** Demo 2 (Semantic Caching)

> **Hint:** ChromaDB stores documents as embeddings. Use `collection.add()` to index and `collection.query()` to search.

Build a `TieredCache` class with two cache tiers plus an LLM fallback:

1. **Tier 1 -- Exact match (hash-based, O(1)):** Normalize the prompt (lowercase, strip whitespace), compute its MD5 hash, and look up a dict. Instant hit for repeated identical queries.
2. **Tier 2 -- Semantic match (embedding-based):** Embed the prompt and compare cosine similarity against all cached embeddings. If the best match exceeds `semantic_threshold`, return the cached response.
3. **Tier 3 -- LLM call (cache miss):** Call the LLM, then store the response in both the exact-match dict and the semantic cache list.

**Requirements:**
- `query(prompt)` returns a dict with keys: `content`, `tier` (`"exact"` / `"semantic"` / `"llm"`), and `cached` (bool)
- Track hit counts per tier in a `stats` dict: `{"exact_hits": 0, "semantic_hits": 0, "misses": 0}`
- When a semantic hit occurs, also add it to the exact cache for future O(1) lookups
- Do NOT cache error responses

### Step-by-Step Plan

1. Implement `_hash(text)`: normalize (lowercase + strip) then MD5 hex digest
2. Implement `_embed(text)`: call the OpenAI embedding API (see Demo 2a for the exact call)
3. Implement `_cosine_sim(a, b)`: dot product divided by product of norms (provided)
4. Implement `query(prompt)`:
   - Step 4a: Compute hash, check `self.exact_cache` dict
   - Step 4b: If no exact hit, embed the prompt and scan `self.semantic_cache`
   - Step 4c: If no semantic hit, call the LLM, store in both caches
5. Run the test block and verify output matches expected

**Hint from Demo 2:** The embedding call is:
```python
self.embed_client.embeddings.create(
    model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
    input=[text]
).data[0].embedding
```

In [ ]:
# Exercise 1 -- Build a Multi-Tier Caching System

class TieredCache:
    def __init__(self, semantic_threshold=0.92):
        self.exact_cache = {}          # hash -> response string
        self.semantic_cache = []       # list of (embedding, query, response)
        self.threshold = semantic_threshold
        self.embed_client = openai.OpenAI()
        self.llm = ChatOpenAI(
            model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
            temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
        )
        self.stats = {"exact_hits": 0, "semantic_hits": 0, "misses": 0}

    def _hash(self, text):
        """Normalize and hash the prompt for exact-match lookup. (Provided)"""
        import hashlib
        return hashlib.md5(text.strip().lower().encode()).hexdigest()

    # NOTE: _hash is provided above. Focus on implementing the query() method below.r().encode()).hexdigest()
        pass

    def _embed(self, text):
        """Return the embedding vector for a single text string."""
        # TODO: call self.embed_client.embeddings.create(...)
        # See Demo 2a for the exact pattern
        pass

    def _cosine_sim(self, a, b):
        """Compute cosine similarity between two vectors."""
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    def query(self, prompt):
        """Check exact cache -> semantic cache -> LLM. Return dict with content, tier, cached."""
        # TODO Step 4a: Tier 1 -- Exact match
        # - Compute the hash of the prompt
        # - If hash exists in self.exact_cache:
        #     - Increment self.stats["exact_hits"]
        #     - Return {"content": self.exact_cache[h], "tier": "exact", "cached": True}

        # TODO Step 4b: Tier 2 -- Semantic match
        # - Embed the prompt using self._embed(prompt)
        # - Loop over self.semantic_cache to find the best cosine similarity
        # - If best_sim >= self.threshold:
        #     - Increment self.stats["semantic_hits"]
        #     - Store in exact_cache too: self.exact_cache[h] = best_response
        #     - Return {"content": best_response, "tier": "semantic", "similarity": best_sim, "cached": True}

        # TODO Step 4c: Tier 3 -- LLM call (cache miss)
        # - Call self.llm.invoke([HumanMessage(content=prompt)]).content
        # - Increment self.stats["misses"]
        # - Store in exact_cache: self.exact_cache[h] = response
        # - Store in semantic_cache: self.semantic_cache.append((embedding, prompt, response))
        # - Return {"content": response, "tier": "llm", "cached": False}

        pass

### Progressive Hints

<details>
<summary>Hint 1: _hash implementation</summary>

```python
def _hash(self, text):
    return hashlib.md5(text.strip().lower().encode()).hexdigest()
```
</details>

<details>
<summary>Hint 2: _embed implementation</summary>

```python
def _embed(self, text):
    return self.embed_client.embeddings.create(
        model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
        input=[text]
    ).data[0].embedding
```
</details>

<details>
<summary>Hint 3: Tier 1 exact match</summary>

```python
h = self._hash(prompt)
if h in self.exact_cache:
    self.stats["exact_hits"] += 1
    return {"content": self.exact_cache[h], "tier": "exact", "cached": True}
```
</details>

<details>
<summary>Hint 4: Tier 2 semantic match</summary>

```python
query_emb = self._embed(prompt)
best_sim, best_response = 0, None
for cached_emb, cached_query, cached_response in self.semantic_cache:
    sim = self._cosine_sim(query_emb, cached_emb)
    if sim > best_sim:
        best_sim = sim
        best_response = cached_response

if best_sim >= self.threshold:
    self.stats["semantic_hits"] += 1
    self.exact_cache[h] = best_response
    return {"content": best_response, "tier": "semantic", "similarity": best_sim, "cached": True}
```
</details>

<details>
<summary>Hint 5: Tier 3 LLM call</summary>

```python
response = self.llm.invoke([HumanMessage(content=prompt)]).content
self.stats["misses"] += 1
self.exact_cache[h] = response
self.semantic_cache.append((query_emb, prompt, response))
return {"content": response, "tier": "llm", "cached": False}
```
Note: reuse `query_emb` from Tier 2 -- no need to embed again.
</details>

In [ ]:
# Test your TieredCache implementation
# Uncomment once you have completed the TODOs above

# cache = TieredCache(semantic_threshold=0.90)
#
# test_queries = [
#     "What are the key trends in healthcare M&A?",                             # LLM (miss)
#     "What are the key trends in healthcare M&A?",                             # Exact hit
#     "What is driving mergers and acquisitions in the healthcare sector?",     # Semantic hit
#     "What are the main cost reduction levers in automotive manufacturing?",   # LLM (miss)
#     "What are the main cost reduction levers in automotive manufacturing?",   # Exact hit
# ]
#
# for q in test_queries:
#     result = cache.query(q)
#     tier_info = f"tier={result['tier']}"
#     if 'similarity' in result:
#         tier_info += f" sim={result['similarity']:.3f}"
#     print(f"[{tier_info:25s}] {q[:50]:50s} | {result['content'][:50]}...")
#
# print(f"\nCache Stats: {cache.stats}")

### Expected Output

```
[tier=llm                    ] What are the key trends in healthcare M&A?         | Key trends in healthcare M&A include: 1. Consol...
[tier=exact                  ] What are the key trends in healthcare M&A?         | Key trends in healthcare M&A include: 1. Consol...
[tier=semantic sim=0.922     ] What is driving mergers and acquisitions in the he  | Key trends in healthcare M&A include: 1. Consol...
[tier=llm                    ] What are the main cost reduction levers in automot  | In automotive manufacturing, key cost reduction ...
[tier=exact                  ] What are the main cost reduction levers in automot  | In automotive manufacturing, key cost reduction ...

Cache Stats: {'exact_hits': 2, 'semantic_hits': 1, 'misses': 2}
```

*The semantic similarity score may vary but should be above your threshold (0.90) for the healthcare M&A paraphrase. The key verification points are: query 2 hits exact cache, query 3 hits semantic cache, and the stats show 2 exact hits, 1 semantic hit, 2 misses.*

---
## Exercise 2 (Easy): Request Validator with Error Recovery

**Reference:** Demo 1 (AI Service) + Demo 3 (Monitoring)

> **Hint:** Validate inputs first, then try the LLM call in a try/except block. On failure, simplify the prompt and retry once.

Build a `SafeAIService` class that:
1. Validates requests using the same rules as Demo 1 (temperature 0-2, max_tokens 1-4096, valid service_type)
2. Calls the LLM
3. If the LLM call fails, retries ONCE with a simpler prompt (shorter, lower temperature)
4. Logs all attempts (success, failure, retry) with structured metadata

**Requirements:**
- `process(prompt, service_type, max_tokens, temperature)` returns a dict with `content`, `status` ("success" / "retry_success" / "failed"), `attempts`, and `log`
- The retry prompt should be: `f"In one sentence: {prompt}"`
- The retry uses temperature=0 and max_tokens=100
- Track all attempts in a log list

In [ ]:
# Exercise 2 -- Request Validator with Error Recovery
# Hint: _validate checks 4 conditions. _call_llm wraps the API call.
# process() calls _validate, then _call_llm, and retries once on failure.

class SafeAIService:
    """AI service with validation, retry logic, and structured logging."""
    
    VALID_SERVICE_TYPES = ["knowledge_search", "engagement_assistant", "briefing_generator"]
    
    def __init__(self):
        self.client = openai.OpenAI()
        self.attempt_log = []  # stores all attempts across all calls
    
    def _validate(self, prompt, service_type, max_tokens, temperature):
        """Validate request parameters. Return (is_valid, error_message)."""
        # TODO: Check that:
        # - prompt is a non-empty string
        # - service_type is in VALID_SERVICE_TYPES
        # - max_tokens is between 1 and 4096
        # - temperature is between 0 and 2
        # Return (True, None) if valid, (False, "error description") if not
        pass
    
    def _call_llm(self, prompt, model="gpt-4o-mini", max_tokens=150, temperature=0.7):
        """Make an LLM call. Return (content, tokens_used) or raise Exception."""
        # TODO: Call self.client.chat.completions.create(...)
        # Return (response.choices[0].message.content, response.usage.total_tokens)
        pass
    
    def _log_attempt(self, prompt, status, content="", error="", tokens=0, latency_ms=0):
        """Log a single attempt with structured metadata."""
        entry = {
            "timestamp": datetime.now().isoformat(),
            "prompt_preview": prompt[:80],
            "status": status,
            "content_preview": content[:80] if content else "",
            "error": error,
            "tokens": tokens,
            "latency_ms": latency_ms
        }
        self.attempt_log.append(entry)
        return entry
    
    def process(self, prompt, service_type="knowledge_search", max_tokens=150, temperature=0.7):
        """Process a request with validation and retry logic."""
        # TODO Step 1: Validate the request
        # - Call self._validate(...)
        # - If invalid, log the attempt with status="validation_error" and return:
        #   {"content": "", "status": "failed", "attempts": 0, "error": error_msg}
        
        # TODO Step 2: First LLM attempt
        # - start = time.time()
        # - Try self._call_llm(prompt, max_tokens=max_tokens, temperature=temperature)
        # - If success: log with status="success", return:
        #   {"content": content, "status": "success", "attempts": 1, "tokens": tokens}
        
        # TODO Step 3: If first attempt raised an exception, retry with simpler prompt
        # - retry_prompt = f"In one sentence: {prompt}"
        # - Log the first failure with status="first_attempt_failed"
        # - Try self._call_llm(retry_prompt, max_tokens=100, temperature=0)
        # - If success: log with status="retry_success", return:
        #   {"content": content, "status": "retry_success", "attempts": 2, "tokens": tokens}
        # - If retry also fails: log with status="retry_failed", return:
        #   {"content": "", "status": "failed", "attempts": 2, "error": str(e)}
        
        pass

In [ ]:
# Test your SafeAIService implementation
# Uncomment once you have completed the TODOs above

# safe_service = SafeAIService()
#
# # Test 1: Valid request
# result = safe_service.process(
#     "What is McKinsey's 7S framework?",
#     service_type="knowledge_search"
# )
# print(f"Test 1 - Valid request:")
# print(f"  Status: {result['status']}, Attempts: {result['attempts']}")
# print(f"  Content: {result['content'][:80]}...")
#
# # Test 2: Invalid service_type
# result = safe_service.process(
#     "Test query",
#     service_type="invalid_type"
# )
# print(f"\nTest 2 - Invalid service_type:")
# print(f"  Status: {result['status']}, Error: {result.get('error', 'none')}")
#
# # Test 3: Invalid temperature
# result = safe_service.process(
#     "Test query",
#     temperature=5.0
# )
# print(f"\nTest 3 - Invalid temperature:")
# print(f"  Status: {result['status']}, Error: {result.get('error', 'none')}")
#
# # Test 4: Empty prompt
# result = safe_service.process("", service_type="knowledge_search")
# print(f"\nTest 4 - Empty prompt:")
# print(f"  Status: {result['status']}, Error: {result.get('error', 'none')}")
#
# print(f"\nTotal logged attempts: {len(safe_service.attempt_log)}")

### Expected Output

```
Test 1 - Valid request:
  Status: success, Attempts: 1
  Content: McKinsey's 7S Framework is a management model developed by Tom Peters and Rober...

Test 2 - Invalid service_type:
  Status: failed, Error: service_type must be one of ['knowledge_search', 'engagement_assistant', 'briefing_generator']

Test 3 - Invalid temperature:
  Status: failed, Error: temperature must be between 0 and 2

Test 4 - Empty prompt:
  Status: failed, Error: prompt must be a non-empty string

Total logged attempts: 2
```

*Note: Only successful LLM calls and first-attempt failures produce log entries. Validation failures are caught before any LLM call.*

---
## Exercise 3 (Easy): Smart Model Router with Cost Tracking

**Reference:** Demo 4 (Model Routing) + Demo 5 (Cost Tracking)

> **Hint:** Conditional routing uses `add_conditional_edges` -- the routing function returns a string that maps to the next node name.

Build a system that:
1. Classifies query complexity (simple / medium / complex)
2. Routes to the appropriate model
3. Tracks costs per query
4. Generates a spending report

**Requirements:**
- `SmartRouter` class with `process(query)` method
- Returns a dict with: `content`, `complexity`, `model`, `cost`, `tokens`
- `get_report()` returns total cost, cost by model, cost by complexity level
- Use the same pricing as Demo 5: gpt-4o-mini input=$0.00015/1k, output=$0.0006/1k; gpt-4o input=$0.005/1k, output=$0.015/1k

In [ ]:
# Exercise 3 -- Smart Model Router with Cost Tracking
# Hint: _classify uses LLM to return "simple"/"medium"/"complex".
# process() classifies, picks model from MODEL_MAP, calls API, tracks cost.

class SmartRouter:
    """Routes queries by complexity and tracks costs."""
    
    PRICING = {
        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
        "gpt-4o": {"input": 0.005, "output": 0.015}
    }
    
    MODEL_MAP = {
        "simple": "gpt-4o-mini",
        "medium": "gpt-4o-mini",
        "complex": "gpt-4o"
    }
    
    def __init__(self):
        self.client = openai.OpenAI()
        self.classifier = ChatOpenAI(
            model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
            temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
        )
        self.records = []  # list of {complexity, model, input_tokens, output_tokens, cost}
    
    def _classify(self, query):
        """Classify query complexity. Return 'simple', 'medium', or 'complex'."""
        # TODO: Use self.classifier to classify the query
        # Use the same system prompt as Demo 4a:
        # "Classify this consulting query's complexity. Return ONLY one word: simple/medium/complex"
        # If the response is not in MODEL_MAP, default to "medium"
        pass
    
    def _calculate_cost(self, model, input_tokens, output_tokens):
        """Calculate cost for a given model and token counts."""
        # TODO: Look up pricing for the model
        # cost = (input_tokens * pricing["input"] + output_tokens * pricing["output"]) / 1000
        pass
    
    def process(self, query):
        """Classify, route, call LLM, track cost. Return result dict."""
        # TODO Step 1: Classify complexity
        # complexity = self._classify(query)
        
        # TODO Step 2: Select model based on complexity
        # model = self.MODEL_MAP[complexity]
        
        # TODO Step 3: Call LLM with the selected model
        # response = self.client.chat.completions.create(
        #     model=model,
        #     messages=[{"role": "user", "content": query}],
        #     max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "150"))
        # )
        
        # TODO Step 4: Calculate cost
        # input_tokens = response.usage.prompt_tokens
        # output_tokens = response.usage.completion_tokens
        # cost = self._calculate_cost(model, input_tokens, output_tokens)
        
        # TODO Step 5: Record the transaction
        # self.records.append({"complexity": complexity, "model": model,
        #                      "input_tokens": input_tokens, "output_tokens": output_tokens,
        #                      "cost": cost})
        
        # TODO Step 6: Return result
        # return {"content": response.choices[0].message.content,
        #         "complexity": complexity, "model": model,
        #         "cost": cost, "tokens": input_tokens + output_tokens}
        pass
    
    def get_report(self):
        """Generate a spending report."""
        # TODO: Calculate:
        # - total_cost: sum of all costs
        # - total_requests: number of records
        # - by_model: {model: {"requests": N, "cost": X}} for each model
        # - by_complexity: {complexity: {"requests": N, "cost": X}} for each level
        # Return as a dict
        pass

In [ ]:
# Test your SmartRouter implementation
# Uncomment once you have completed the TODOs above

# router = SmartRouter()
#
# test_queries = [
#     "What does MECE stand for?",
#     "List the Big 4 consulting firms.",
#     "Summarize trends in European fintech regulation.",
#     "Design a post-merger integration plan for a $2B healthcare acquisition with synergy capture, org design, and Day 1 readiness.",
#     "What is Porter's Five Forces?",
# ]
#
# for q in test_queries:
#     result = router.process(q)
#     print(f"[{result['complexity']:7s}] [{result['model']:12s}] ${result['cost']:.6f} | {q[:55]}")
#
# print("\n--- Spending Report ---")
# report = router.get_report()
# print(f"Total cost: ${report['total_cost']:.6f}")
# print(f"Total requests: {report['total_requests']}")
# print("\nBy model:")
# for model, stats in report['by_model'].items():
#     print(f"  {model}: {stats['requests']} reqs, ${stats['cost']:.6f}")
# print("\nBy complexity:")
# for level, stats in report['by_complexity'].items():
#     print(f"  {level}: {stats['requests']} reqs, ${stats['cost']:.6f}")

### Expected Output

```
[simple ] [gpt-4o-mini ] $0.000052 | What does MECE stand for?
[simple ] [gpt-4o-mini ] $0.000061 | List the Big 4 consulting firms.
[medium ] [gpt-4o-mini ] $0.000098 | Summarize trends in European fintech regulation.
[complex] [gpt-4o      ] $0.003850 | Design a post-merger integration plan for a $2B healthcare...
[simple ] [gpt-4o-mini ] $0.000048 | What is Porter's Five Forces?

--- Spending Report ---
Total cost: $0.004109
Total requests: 5

By model:
  gpt-4o-mini: 4 reqs, $0.000259
  gpt-4o: 1 reqs, $0.003850

By complexity:
  simple: 3 reqs, $0.000161
  medium: 1 reqs, $0.000098
  complex: 1 reqs, $0.003850
```

*Exact costs will vary based on token counts. The key verification is that simple/medium queries route to gpt-4o-mini and complex queries route to gpt-4o, with gpt-4o being significantly more expensive per query.*

---
---
# Optional Exercises


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
These exercises are at an intermediate level and build on the core exercises above. Attempt them if you finish early or want additional practice.

---
## Optional Exercise 1 (Intermediate): Cache with TTL and Eviction

**Reference:** Demo 2 (Semantic Caching)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
Extend the TieredCache from Exercise 1 with:
1. **Time-to-live (TTL):** Each cached entry expires after `ttl_seconds` (default 300). Expired entries are not returned as hits.
2. **Max cache size with LRU eviction:** When the semantic cache exceeds `max_size` entries, evict the least-recently-used entry.

**Requirements:**
- Store timestamps with each cache entry
- `_is_expired(entry)` checks if TTL has passed
- When cache is full and a new entry must be added, remove the oldest entry
- `get_cache_info()` returns: total entries, expired entries, cache utilization (%)

In [ ]:
# Optional Exercise 1 -- Cache with TTL and Eviction

class TTLCache:
    """Semantic cache with time-to-live and LRU eviction."""
    
    def __init__(self, semantic_threshold=0.92, ttl_seconds=300, max_size=100):
        self.exact_cache = {}          # hash -> {"response": str, "timestamp": datetime, "last_used": datetime}
        self.semantic_cache = []       # list of {"embedding": [...], "query": str, "response": str, "timestamp": datetime, "last_used": datetime}
        self.threshold = semantic_threshold
        self.ttl_seconds = ttl_seconds
        self.max_size = max_size
        self.embed_client = openai.OpenAI()
        self.llm = ChatOpenAI(
            model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
            temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
        )
        self.stats = {"exact_hits": 0, "semantic_hits": 0, "misses": 0, "expired": 0, "evictions": 0}

    def _is_expired(self, entry):
        """Check if a cache entry has exceeded its TTL."""
        # TODO: Compare (datetime.now() - entry["timestamp"]).total_seconds() > self.ttl_seconds
        pass

    def _evict_lru(self):
        """Remove the least-recently-used entry from semantic_cache."""
        # TODO: Find the entry with the oldest "last_used" timestamp and remove it
        # Increment self.stats["evictions"]
        pass

    def _hash(self, text):
        return hashlib.md5(text.strip().lower().encode()).hexdigest()

    def _embed(self, text):
        return self.embed_client.embeddings.create(
            model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
            input=[text]
        ).data[0].embedding

    def _cosine_sim(self, a, b):
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    def query(self, prompt):
        """Query with TTL checking and LRU eviction."""
        # TODO: Same as Exercise 1, but:
        # - Check _is_expired() before returning a hit
        # - If expired, increment stats["expired"] and skip it
        # - Update "last_used" on any hit
        # - Call _evict_lru() before adding if len(semantic_cache) >= max_size
        pass

    def get_cache_info(self):
        """Return cache status information."""
        # TODO: Return dict with:
        # - total_entries: len(semantic_cache)
        # - expired_entries: count entries where _is_expired is True
        # - cache_utilization: len(semantic_cache) / max_size * 100
        # - stats: self.stats
        pass


# Test (uncomment when implemented)
# ttl_cache = TTLCache(semantic_threshold=0.90, ttl_seconds=5, max_size=3)
# 
# # Add entries
# result1 = ttl_cache.query("What is MECE?")
# result2 = ttl_cache.query("What is Porter's Five Forces?")
# result3 = ttl_cache.query("What is the BCG matrix?")
# print(f"Cache info after 3 entries: {ttl_cache.get_cache_info()}")
#
# # This should trigger eviction (max_size=3)
# result4 = ttl_cache.query("What is the Ansoff matrix?")
# print(f"Cache info after 4th entry (eviction): {ttl_cache.get_cache_info()}")
#
# # Wait for TTL to expire
# import time
# time.sleep(6)
# result_expired = ttl_cache.query("What is MECE?")  # Should miss (expired)
# print(f"After TTL expiry - tier: {result_expired.get('tier', 'unknown')}")

---
## Optional Exercise 2 (Intermediate): Full Monitoring Dashboard

**Reference:** Demo 3 (Monitoring) + Demo 5 (Cost Tracking)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
Build a monitor that tracks:
1. **Latency percentiles:** p50, p90, p95, p99
2. **Error rates:** per model, per service type
3. **Cost per engagement:** total spending attributed to each engagement ID
4. **SLA alerts:** Generate alerts when latency exceeds a threshold or error rate exceeds a limit

**Requirements:**
- `MonitoringDashboard` class
- `record(request_id, engagement_id, model, tokens, latency_ms, cost, status)` logs a request
- `check_sla(latency_threshold_ms=5000, error_rate_threshold=0.05)` returns a list of alert strings
- `get_dashboard()` returns a comprehensive dict with all metrics

In [ ]:
# Optional Exercise 2 -- Full Monitoring Dashboard

class MonitoringDashboard:
    """Comprehensive monitoring with SLA alerting."""
    
    def __init__(self):
        self.records = []
    
    def record(self, request_id, engagement_id, model, tokens, latency_ms, cost, status="success"):
        """Log a single request."""
        # TODO: Append a dict with all fields plus a timestamp to self.records
        pass
    
    def _latency_percentiles(self):
        """Calculate p50, p90, p95, p99 latency."""
        # TODO: Use np.percentile on the latency values from self.records
        # Return {"p50": ..., "p90": ..., "p95": ..., "p99": ...}
        pass
    
    def _error_rates(self):
        """Calculate error rates per model and per service type."""
        # TODO: For each unique model, calculate (errors / total) for that model
        # Return {"overall": ..., "by_model": {...}}
        pass
    
    def _cost_per_engagement(self):
        """Calculate total cost attributed to each engagement."""
        # TODO: Group records by engagement_id, sum costs
        # Return {engagement_id: total_cost, ...}
        pass
    
    def check_sla(self, latency_threshold_ms=5000, error_rate_threshold=0.05):
        """Check SLA thresholds and return alert messages."""
        # TODO: Check if p95 latency exceeds latency_threshold_ms
        # Check if overall error rate exceeds error_rate_threshold
        # Return a list of alert strings (empty if all SLAs are met)
        pass
    
    def get_dashboard(self):
        """Return full dashboard metrics."""
        # TODO: Return {
        #   "total_requests": ...,
        #   "latency_percentiles": self._latency_percentiles(),
        #   "error_rates": self._error_rates(),
        #   "cost_per_engagement": self._cost_per_engagement(),
        #   "total_cost": sum of all costs,
        #   "sla_alerts": self.check_sla()
        # }
        pass


# Test (uncomment when implemented)
# dashboard = MonitoringDashboard()
#
# # Simulate 10 requests
# import random
# for i in range(10):
#     dashboard.record(
#         request_id=f"req-{i:03d}",
#         engagement_id=random.choice(["ENG-001", "ENG-002", "ENG-003"]),
#         model=random.choice(["gpt-4o-mini", "gpt-4o-mini", "gpt-4o"]),
#         tokens=random.randint(100, 500),
#         latency_ms=random.uniform(1000, 8000),
#         cost=random.uniform(0.0001, 0.005),
#         status=random.choice(["success"] * 9 + ["error"])
#     )
#
# result = dashboard.get_dashboard()
# print("--- Dashboard ---")
# for k, v in result.items():
#     print(f"  {k}: {v}")
#
# alerts = dashboard.check_sla(latency_threshold_ms=5000, error_rate_threshold=0.05)
# print(f"\nSLA Alerts: {alerts if alerts else 'None -- all SLAs met'}")

---
## Optional Exercise 3 (Intermediate): Production AI Service

This exercise is intentionally challenging and combines all 5 demo patterns.


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
Build a `ProductionAIService` class that integrates:
1. **Request validation** (Demo 1): Reject invalid requests before any LLM call
2. **Semantic caching** (Demo 2): Check cache before calling the LLM
3. **Model routing** (Demo 4): Route cache misses to the appropriate model based on complexity
4. **Monitoring** (Demo 3): Log every request (cached or not) with structured metadata
5. **Cost tracking** (Demo 5): Track costs and raise alerts at budget thresholds

**Request flow:**
```
Request -> Validate -> Check Cache -> [HIT] -> Log + Return
                                   -> [MISS] -> Classify Complexity -> Route to Model -> Call LLM -> Cache Response -> Track Cost -> Log -> Return
```

**Requirements:**
- Single `process(prompt, engagement_id)` method
- Returns dict with: `content`, `source` ("cache" or "llm"), `model`, `complexity`, `cost`, `latency_ms`
- `get_status()` returns: cache hit rate, total cost, budget remaining, request count, model distribution

In [ ]:
# Optional Exercise 3 -- Production AI Service (Combines ALL 5 Demo Patterns)

class ProductionAIService:
    """Full production AI service combining validation, caching, routing, monitoring, and cost tracking."""
    
    PRICING = {
        "gpt-4o-mini": {"input": 0.00015, "output": 0.0006},
        "gpt-4o": {"input": 0.005, "output": 0.015}
    }
    
    MODEL_MAP = {
        "simple": "gpt-4o-mini",
        "medium": "gpt-4o-mini",
        "complex": "gpt-4o"
    }
    
    def __init__(self, similarity_threshold=0.90, budget_limit=1.0):
        self.client = openai.OpenAI()
        self.classifier = ChatOpenAI(
            model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
            temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
        )
        # Cache state
        self.cache = []  # list of (embedding, query, response)
        self.threshold = similarity_threshold
        # Monitoring state
        self.logs = []
        # Cost state
        self.budget_limit = budget_limit
        self.total_cost = 0.0
        # Stats
        self.stats = {"cache_hits": 0, "cache_misses": 0, "total_requests": 0, "errors": 0}

    def _validate(self, prompt, engagement_id):
        """Validate the request. Return (True, None) or (False, error_msg)."""
        # TODO: Check prompt is non-empty string, engagement_id is non-empty
        pass

    def _embed(self, text):
        """Embed a text string."""
        # TODO: Same as Demo 2
        pass

    def _check_cache(self, prompt):
        """Check semantic cache. Return (hit: bool, response: str, similarity: float)."""
        # TODO: Embed prompt, scan cache, return best match if above threshold
        pass

    def _classify_complexity(self, query):
        """Classify query complexity."""
        # TODO: Same as Demo 4
        pass

    def _calculate_cost(self, model, input_tokens, output_tokens):
        """Calculate cost."""
        # TODO: Same as Demo 5
        pass

    def process(self, prompt, engagement_id="ENG-DEFAULT"):
        """Full production request flow: validate -> cache -> route -> call -> log -> return."""
        start = time.time()
        self.stats["total_requests"] += 1
        
        # TODO Step 1: Validate
        # If invalid, return {"content": "", "source": "error", "error": msg, ...}
        
        # TODO Step 2: Check cache
        # If hit, log it and return {"content": ..., "source": "cache", "model": "cached",
        #                             "complexity": "cached", "cost": 0, "latency_ms": ...}
        
        # TODO Step 3: Classify complexity and select model
        
        # TODO Step 4: Call LLM
        
        # TODO Step 5: Calculate cost and track budget
        
        # TODO Step 6: Cache the response
        
        # TODO Step 7: Log and return full result dict
        
        pass

    def get_status(self):
        """Return service status summary."""
        # TODO: Return dict with:
        # - cache_hit_rate: cache_hits / total_requests (or 0 if no requests)
        # - total_cost
        # - budget_remaining: budget_limit - total_cost
        # - total_requests
        # - model_distribution: count of requests per model from logs
        # - stats: self.stats
        pass


# Test (uncomment when implemented)
# production = ProductionAIService(similarity_threshold=0.90, budget_limit=0.10)
#
# queries = [
#     ("What is MECE?", "ENG-001"),
#     ("Design a market entry strategy for Southeast Asian e-commerce.", "ENG-002"),
#     ("What does MECE mean in consulting?", "ENG-001"),  # should hit cache
#     ("What is Porter's Five Forces?", "ENG-003"),
#     ("Develop a post-merger integration playbook.", "ENG-002"),
# ]
#
# for prompt, eng_id in queries:
#     result = production.process(prompt, engagement_id=eng_id)
#     print(f"[{result.get('source', '?'):6s}] [{result.get('complexity', '?'):7s}] "
#           f"${result.get('cost', 0):.6f} | {prompt[:50]}")
#
# print("\n--- Service Status ---")
# status = production.get_status()
# for k, v in status.items():
#     print(f"  {k}: {v}")

### Expected Output (approximate)

```
[llm   ] [simple ] $0.000045 | What is MECE?
[llm   ] [complex] $0.003920 | Design a market entry strategy for Southeast Asian...
[cache ] [cached ] $0.000000 | What does MECE mean in consulting?
[llm   ] [simple ] $0.000052 | What is Porter's Five Forces?
[llm   ] [complex] $0.004100 | Develop a post-merger integration playbook.

--- Service Status ---
  cache_hit_rate: 0.2
  total_cost: 0.008117
  budget_remaining: 0.091883
  total_requests: 5
  model_distribution: {'gpt-4o-mini': 2, 'gpt-4o': 2, 'cached': 1}
  stats: {'cache_hits': 1, 'cache_misses': 4, 'total_requests': 5, 'errors': 0}
```

*The third query ("What does MECE mean in consulting?") should hit the cache because it is semantically similar to "What is MECE?". Costs will vary but the pattern should be: simple queries are cheap, complex queries are expensive, cached queries are free.*

---
## Summary

| Exercise | Patterns Combined | Difficulty |
|----------|------------------|------------|
| 1. Multi-Tier Cache | Exact + Semantic caching | Core |
| 2. Safe AI Service | Validation + Retry + Logging | Core |
| 3. Smart Router | Routing + Cost Tracking | Core |
| Opt 1. TTL Cache | Caching + Eviction | Medium |
| Opt 2. Monitoring Dashboard | Monitoring + Alerting | Medium |
| Opt 3. Production Service | ALL 5 patterns | Challenging |

When you have completed Exercises 1-3, you have the core building blocks. The optional exercises show how they compose into a real production system.